In [1]:
%pylab inline 
import torch 
import torch.nn as nn 
import torch.nn.functional as F
from tqdm import trange 
from utils import *

Populating the interactive namespace from numpy and matplotlib


In [2]:
class LSTMCell(nn.Module):
    def __init__(self, in_size, h_size):
        super(LSTMCell, self).__init__()
        self.h_size = h_size
        self.fg = nn.Linear(in_size + h_size, h_size)
        self.in_g = nn.Linear(in_size + h_size, h_size)
        self.out_g = nn.Linear(in_size + h_size, h_size)
        self.c_g = nn.Linear(in_size + h_size, h_size)
    def forward(self, x, h, c):
        combined = torch.cat((x, h), 1)
        c = (self.fg(combined) * c) + (self.in_g(combined) + F.tanh(self.c_g(combined)))
        out_gate = self.out_g(combined)
        h = out_gate * F.tanh(c)
        return (h, c)
    
class LSTM(nn.Module):
    def __init__(self, in_size, h_size, out_size):
        super(LSTM, self).__init__()
        self.h_size = h_size
        self.lstm = LSTMCell(in_size, h_size)
        self.class_layer = nn.Linear(h_size, out_size)
    
    def forward(self, x, h, c): # Init a c and h when instantiating 
        h_out, c_out = self.lstm(x, h, c)
        out = self.class_layer(h_out)
        return out, h_out, c_out  
    
    def init_h_states(self):
        return torch.zeros(1, self.h_size)

lstm = LSTM(n_chars,128, n_categs)


In [3]:
# Training Function 
opt = torch.optim.Adam(lstm.parameters())
loss_function = nn.CrossEntropyLoss()
def train(categ_tensor, line_tensor):
    h = lstm.init_h_states()
    c = lstm.init_h_states()
    
    for i in range(line_tensor.size(0)):
        out, h, c = lstm(line_tensor[i], h, c)
    loss = loss_function(out, categ_tensor)
    # Update Weights 
    opt.zero_grad()
    loss.backward()
    opt.step()
    return out, loss.item()

In [4]:
epochs = 100000
steps = epochs / 5 
losses = []
c_loss = 0
for i in trange(epochs+1):
    categ, line, categ_tensor, line_tensor = gen_rand_training_example()
    out, loss = train(categ_tensor, line_tensor)
    c_loss += loss 

    if i % steps == 0:
        guess, guess_i = categ_from_out(out)
        correct = "CORRECT" if guess == categ else f"WRONG ({categ})"
        print(f"loss = {loss:.3f} {guess} / {correct}")



    


  1%|          | 63/10001 [00:00<00:30, 320.69it/s]tensor([[0.0832]], grad_fn=<TopkBackward>)
loss = 2.876 German / WRONG (Irish)
 21%|██        | 2070/10001 [00:05<00:21, 360.50it/s]tensor([[2.0989]], grad_fn=<TopkBackward>)
loss = 1.430 Spanish / WRONG (Greek)
 40%|████      | 4039/10001 [00:10<00:15, 376.71it/s]tensor([[3.5015]], grad_fn=<TopkBackward>)
loss = 2.488 Spanish / WRONG (Italian)
 61%|██████    | 6053/10001 [00:16<00:10, 368.17it/s]tensor([[2.7783]], grad_fn=<TopkBackward>)
loss = 0.861 Russian / CORRECT
 81%|████████  | 8069/10001 [00:21<00:05, 372.22it/s]tensor([[3.1030]], grad_fn=<TopkBackward>)
loss = 1.242 Chinese / WRONG (Vietnamese)
100%|██████████| 10001/10001 [00:27<00:00, 370.35it/s]tensor([[0.2900]], grad_fn=<TopkBackward>)
loss = 2.720 Irish / WRONG (Portuguese)

